In [ ]:
!pip install openai

In [ ]:
#@title 오늘 날짜 말하기 채점 프롬프트

import re
from google.colab import userdata
from openai import OpenAI


# =========================
# 기본 설정
# =========================

API_KEY = userdata.get("OPENAI_API_KEY")

if not API_KEY:
    raise ValueError("OPENAI_API_KEY가 없습니다. 코랩 Secrets를 확인하세요.")

client = OpenAI(api_key=API_KEY)

MODEL_NAME = "gpt-5.4-mini-2026-03-17"

VALID_SCORES = {0, 40, 60, 80, 100}


# =========================
# 여기부터 직접 입력하는 부분
# =========================

CRITERIA_PROMPT = """
당신은 노인 인지검사의 화행 적절성 채점자다. 아래 문항에 대한 수급자의 음성 답변(STT 변환 텍스트)을 평가한다.

[이 유형의 문항 의도]
현재 날짜, 시간, 요일, 계절, 상대적 시간 정보를 말하도록 요구하는 문항이다. 이 평가는 날짜의 실제 정답 여부를 평가하는 것이 아니라, 문항이 요구한 시간·날짜 요소를 답변이 충족했는지를 평가한다.

[문항]
{question_text}

[답변]
{stt_text}

[평가 규칙]

* 평가 대상은 "답변이 문항이 요구한 시간·날짜 요소를 충족했는가"이다.
* 날짜의 실제 사실 정확성은 평가하지 않는다.
* 발음, 문법, 띄어쓰기, 존댓말 여부는 평가하지 않는다.
* 먼저 [문항]을 읽고, 문항이 요구하는 시간·날짜 요소를 빠짐없이 나눈다.
* 요구 요소에는 연도, 월, 일, 요일, 계절뿐 아니라 오전/오후, 오늘/내일/어제, 며칠 뒤/며칠 전, 주말까지 남은 날 수처럼 문항이 직접 요구한 시간 관련 정보도 포함한다.
* 문항이 여러 요소를 요구하면 각 요소를 따로 세어야 한다.
* 답변이 요구 요소 중 하나만 충족했다고 해서 100점으로 평가하지 않는다.
* 문항이 요구하지 않은 요소가 빠졌다는 이유로 감점하지 않는다.
* 답변이 비었거나 침묵뿐이면 0점이다.
* 답변이 "모른다", "몰라", "내가 어떻게 알아"처럼 시간·날짜 정보를 제공하지 않으면 0점이다.
* 문항과 전혀 무관한 답변이면 0점이다.
* 시간·날짜와 관련된 표현이지만 문항이 요구한 구체 요소를 충족하지 못한 경우는 40점으로 평가한다. 예: 요일을 물었는데 "평일"이라고 답함, 월을 물었는데 "여름"이라고 답함.
* 문항이 "요일"을 요구하는 경우, 월요일·화요일·수요일·목요일·금요일·토요일·일요일처럼 구체적인 요일명을 답해야 해당 요구 요소를 충족한 것으로 본다.
* "평일", "주말", "쉬는 날", "일하는 날"처럼 요일의 범주나 성격만 말한 경우는 구체적인 요일을 답한 것이 아니므로 요일 요구 요소를 충족하지 못한 것으로 본다.
* 단, 문항이 "평일인지 주말인지"를 물은 경우에는 "평일", "주말"을 요구 요소 충족으로 인정한다.
* 문항이 "며칠", "몇 일", "오늘은 며칠"처럼 일자를 요구하는 경우, 답변의 숫자 표현을 일자로 해석할 수 있으면 요구 요소를 충족한 것으로 본다.
* STT 변환 특성상 "이일", "삼일", "사일", "십오일", "이십일일"처럼 숫자와 '일'이 붙어 있거나 띄어쓰기가 불완전해도 일자 표현으로 인정한다.
* 답변이 문항의 요구 요소와 자연스럽게 연결되면, 표기 혼동 가능성만으로 0점 처리하지 않는다.
* 예를 들어 문항이 "오늘은 며칠인가요?"이고 답변이 "이일"이면 "2일"을 답한 것으로 보고 100점으로 평가한다.

[요구 요소 판별 보충 규칙]
- 문항이 요구하는 요소는 문항의 표현을 기준으로 따로 나누어 센다.
- "몇 월 며칠"은 하나의 요소가 아니라 "월"과 "일" 두 요소이다.
- "팔월 이십일", "유월 사일", "8월 21일"처럼 월과 일이 함께 나온 답변은 월 요소와 일 요소를 각각 충족한 것으로 본다.
- "연도와 월, 일", "몇 년 몇 월 며칠", "월, 일, 요일"처럼 나열된 항목은 각각 별도의 요구 요소로 본다.
- "오전인지 오후인지", "평일인지 주말인지", "다음 달", "내일 요일", "며칠 뒤가 주말인지"처럼 문항이 직접 요구한 상대적·생활형 시간 정보도 별도의 요구 요소로 본다.
- 문항이 구체적 요일명을 요구하는 경우, "평일", "주말", "쉬는 날"은 구체적 요일을 답한 것이 아니므로 요일 요소를 충족하지 못한 것으로 본다.
- 단, 문항이 "평일인지 주말인지"를 직접 요구하는 경우에는 "평일", "주말"을 해당 요구 요소 충족으로 인정한다.

[채점 절차]
반드시 다음 순서로 판단한다.

1. [문항]에서 요구 요소를 모두 찾는다.
   예: "오늘이 몇 월 며칠인지, 그리고 오전인지 오후인지 말씀해 주세요."
   → 요구 요소는 "월", "일", "오전/오후"이다. 총 3개이다.

2. [답변]에서 실제로 충족한 요소만 찾는다.
   예: "오후"
   → 충족한 요소는 "오전/오후" 1개뿐이다. "월"과 "일"은 충족하지 못했다.

3. 요구 요소 개수와 충족 요소 개수를 비교해 점수를 정한다.

[점수 기준]

* 100: 문항이 요구한 시간·날짜 요소를 모두 분명히 답함.
* 80: 요구 요소 대부분을 답했으나 일부가 빠졌거나, 한 요소가 불확실함.
* 60: 요구 요소 중 일부만 답했거나, 절반 이하만 분명히 답함.
* 40: 시간·날짜와 관련은 있으나 문항이 요구한 요소를 거의 충족하지 못함.
* 0: 무응답, 침묵, 모름/거부, 알아들을 수 없음, 문항과 무관함.

[생활형·추론형 문항 보충 규칙]
- 문항이 "오늘 날짜/현재 날짜/오늘 요일/이번 달" 같은 기본 시간 요소와 "다음 달", "내일 요일", "며칠 뒤가 주말인지", "평일/주말 여부" 같은 부가·추론 요소를 함께 요구하는 경우가 있다.
- 이런 문항에서 기본 시간 요소를 분명히 답했지만 부가·추론 요소만 빠진 경우는 80점으로 평가한다.
- 반대로 부가·추론 요소만 답하고 기본 시간 요소를 답하지 않은 경우는 60점으로 평가한다.
- 기본 요소와 부가 요소를 모두 답하면 100점이다.

[요구 요소 개수별 기준]

- 요구 요소가 1개인 경우:
  - 요구 요소를 분명히 답하면 100점.
  - 요구 요소를 불확실하게 답하면 60점.
  - 요구 요소를 답하지 못하면 0점.
  - 시간·날짜와 관련은 있으나 요구 요소를 충족하지 못하면 40점.

- 요구 요소가 2개인 경우:
  - 2개를 모두 답하면 100점.
  - 1개만 답한 경우는 아래 기준으로 나눈다.
    1) 기본 날짜·시간 요소를 분명히 답하고 부가 요소만 빠졌으면 80점.
    2) 부가 요소만 답하고 기본 날짜·시간 요소가 빠졌으면 60점.
    3) 두 요소의 중요도 차이가 없고 1개만 답했으면 60점.
  - 시간·날짜 관련 말은 있으나 요구 요소를 충족하지 못하면 40점.
  - 0개를 답하거나 모른다고 하면 0점.

- 요구 요소가 3개인 경우:
  - 3개를 모두 답하면 100점.
  - 2개를 분명히 답하면 80점.
  - 1개만 답하면 60점.
  - 시간·날짜 관련 말은 있으나 요구 요소를 충족하지 못하면 40점.
  - 0개를 답하거나 모른다고 하면 0점.

- 요구 요소가 4개 이상인 경우:
  - 모두 답하면 100점.
  - 1개만 빠지고 나머지를 답하면 80점.
  - 절반 정도 또는 절반 이하만 답하면 60점.
  - 시간·날짜 관련 말은 있으나 요구 요소를 거의 충족하지 못하면 40점.
  - 0개를 답하거나 모른다고 하면 0점.

[추가 경계 규칙]
- "팔월 이십일", "유월 사일", "8월 21일"처럼 월과 일이 붙어 있는 표현은 반드시 월 요소 1개와 일 요소 1개를 각각 충족한 것으로 센다. 절대 하나의 날짜 요소로만 세지 않는다.
- 요구 요소가 3개이고 답변이 그중 2개를 충족하면 80점으로 평가한다.
- 요구 요소가 4개이고 답변이 그중 3개를 충족하면 80점으로 평가한다.

- 생활형·추론형 문항에서 현재 날짜, 현재 요일, 현재 월처럼 기준이 되는 기본 요소를 답했고, 다음 달, 내일 요일, 주말까지 남은 날 수, 평일/주말 여부 같은 부가 요소만 빠진 경우는 80점으로 평가한다.
- 반대로 부가 요소만 답하고 기본 요소가 빠진 경우는 60점으로 평가한다.

- 문항에 나온 단어를 사용했더라도, 요구 요소의 값을 답하지 않으면 충족으로 보지 않는다.
- "주말은 곧 오겠죠", "내일은 내일이지"처럼 문항의 단어를 반복하거나 일반적인 반응만 말하고, 며칠 뒤인지·무슨 요일인지 같은 요구 요소의 값을 답하지 않으면 40점으로 평가한다.

- "아침 먹었어요", "밥 먹었어요", "날씨가 좋아요", "기분이 좋아요"처럼 일상 활동, 식사, 날씨, 기분에 대한 말은 문항이 요구한 시간·날짜 요소와 무관하면 0점으로 평가한다.
- 특히 "오전/오후"를 묻는 문항에서 "아침 먹었어요"처럼 식사 경험을 말한 것은 오전/오후를 직접 답한 것이 아니므로 0점으로 평가한다.

- 요구 요소에 대해 두 개 이상의 후보를 말하며 헷갈린다고 한 경우, 해당 요소를 불확실하게 시도한 것으로 보고 60점 평가에 반영한다.
- 단, 요구 요소가 여러 개인데 한 요소만 불확실하게 답하고 나머지를 답하지 못한 경우는 60점을 넘지 않는다.

[주의할 예시]

* 문항: "오늘이 몇 월 며칠인지, 그리고 오전인지 오후인지 말씀해 주세요."
  답변: "오후"
  판단: 요구 요소는 월, 일, 오전/오후 총 3개이다. 답변은 오전/오후 1개만 충족한다. 따라서 60점이다.

* 문항: "오늘이 며칠인지, 그리고 며칠 뒤가 주말인지 말씀해 주세요."
  답변: "오늘 사일인데, 이틀 뒤가 주말이야"
  판단: 요구 요소는 오늘 며칠, 주말까지 남은 날 수 총 2개이다. 답변은 2개를 모두 충족한다. 따라서 100점이다.

* 문항: "지금 몇 년, 몇 월, 며칠, 무슨 요일인지 말씀해 주세요."
  답변: "몰라"
  판단: 시간·날짜 정보를 제공하지 않았다. 따라서 0점이다.

[출력]
0, 40, 60, 80, 100 중 숫자 하나만 출력한다. 다른 텍스트·기호·설명·코드펜스를 절대 붙이지 않는다.
""".strip()


TEST_CASES = [
    {
        "question_type": "오늘 날짜 말하기",
        "question_text": "오늘은 무슨 요일인가요?",
        "answer": "목요일이여.",
        "expected_score": 100,
    },
    {
        "question_type": "오늘 날짜 말하기",
        "question_text": "이번 달은 몇 월인지 말씀해 주세요.",
        "answer": "유월이랑께.",
        "expected_score": 100,
    },
    {
        "question_type": "오늘 날짜 말하기",
        "question_text": "오늘이 며칠이고 무슨 요일인가요?",
        "answer": "사일인디, 요일은 잘 모르겄어.",
        "expected_score": 60,
    },
    {
        "question_type": "오늘 날짜 말하기",
        "question_text": "오늘 요일을 말씀해 주세요.",
        "answer": "평일이제.",
        "expected_score": 40,
    },
]

# =========================
# 여기까지 직접 입력하는 부분
# =========================


def build_prompt(criteria_prompt, question_text, answer_text):
    """
    프롬프트 안의 {question_text}, {stt_text}를
    실제 문항과 답변으로 바꿔 끼우는 함수
    """
    return (
        criteria_prompt
        .replace("{question_text}", question_text)
        .replace("{stt_text}", answer_text)
    )


def parse_score(raw_output):
    """
    LLM 출력에서 0, 40, 60, 80, 100 중 하나만 추출하는 함수
    """
    text = raw_output.strip()

    if text.isdigit():
        score = int(text)
        if score in VALID_SCORES:
            return score

    numbers = re.findall(r"\b(?:0|40|60|80|100)\b", text)

    if len(numbers) == 1:
        score = int(numbers[0])
        if score in VALID_SCORES:
            return score

    raise ValueError(f"점수 형식이 올바르지 않습니다. LLM 출력: {raw_output}")


def grade_answer(criteria_prompt, question_text, answer_text):
    """
    OpenAI API를 호출해서 화행 적절성 점수를 받는 함수
    """
    prompt = build_prompt(criteria_prompt, question_text, answer_text)

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        temperature=0,
        max_output_tokens=32,
    )

    raw_output = response.output_text.strip()
    score = parse_score(raw_output)

    return score, raw_output


def run_tests():
    for index, case in enumerate(TEST_CASES, start=1):
        question_type = case.get("question_type", "")
        question_text = case["question_text"]
        answer = case["answer"]
        expected_score = case.get("expected_score")

        try:
            score, raw_output = grade_answer(
                criteria_prompt=CRITERIA_PROMPT,
                question_text=question_text,
                answer_text=answer
            )

            print(f"[{index}]")
            print(f"문항 유형: {question_type}")
            print(f"문항: {question_text}")
            print(f"답변: {answer}")

            if expected_score is not None:
                is_match = "일치" if score == expected_score else "불일치"
                print(f"예상 점수: {expected_score}")
                print(f"LLM 점수: {score}")
                print(f"결과: {is_match}")
            else:
                print(f"LLM 점수: {score}")

            print(f"원본 출력: {raw_output}")
            print("-" * 50)

        except Exception as e:
            print(f"[{index}] 오류 발생")
            print(f"문항 유형: {question_type}")
            print(f"문항: {question_text}")
            print(f"답변: {answer}")
            print(f"오류 내용: {e}")
            print("-" * 50)


run_tests()

[1]
문항 유형: 오늘 날짜 말하기
문항: 오늘은 무슨 요일인가요?
답변: 목요일이여.
예상 점수: 100
LLM 점수: 100
결과: 일치
원본 출력: 100
--------------------------------------------------
[2]
문항 유형: 오늘 날짜 말하기
문항: 이번 달은 몇 월인지 말씀해 주세요.
답변: 유월이랑께.
예상 점수: 100
LLM 점수: 100
결과: 일치
원본 출력: 100
--------------------------------------------------
[3]
문항 유형: 오늘 날짜 말하기
문항: 오늘이 며칠이고 무슨 요일인가요?
답변: 사일인디, 요일은 잘 모르겄어.
예상 점수: 60
LLM 점수: 60
결과: 일치
원본 출력: 60
--------------------------------------------------
[4]
문항 유형: 오늘 날짜 말하기
문항: 오늘 요일을 말씀해 주세요.
답변: 평일이제.
예상 점수: 40
LLM 점수: 40
결과: 일치
원본 출력: 40
--------------------------------------------------


In [ ]:
#@title 그림 설명하기 채점 프롬프트

import re
from google.colab import userdata
from openai import OpenAI


# =========================
# 기본 설정
# =========================

API_KEY = userdata.get("OPENAI_API_KEY")

if not API_KEY:
    raise ValueError("OPENAI_API_KEY가 없습니다. 코랩 Secrets를 확인하세요.")

client = OpenAI(api_key=API_KEY)

MODEL_NAME = "gpt-5.4-mini-2026-03-17"

VALID_SCORES = {0, 40, 60, 80, 100}


# =========================
# 여기부터 직접 입력하는 부분
# =========================

CRITERIA_PROMPT = """
당신은 노인 인지검사의 화행 적절성 채점자다. 아래 문항에 대한 수급자의 음성 답변(STT 변환 텍스트)을 평가한다.

[이 유형의 문항 의도]
화면에 제시된 그림을 보고 그 상황을 설명하는 발화 능력 확인. 수급자가 그림 속 사물·인물·행위·장면을 언어로 묘사하도록 요구하는 문항이다.
이 평가는 그림을 얼마나 완벽하게 자세히 설명했는지를 평가하는 것이 아니라, 그림 설명하기라는 말하기 행위를 문항 의도에 맞게 수행했는지를 평가한다.

[그림 설명] {image_description}
[문항] {question_text}
[답변] {stt_text}

[평가 규칙]

* 채점자는 그림을 직접 보지 못한다. 위 [그림 설명]이 그림의 실제 내용이다. 답변이 이 그림 내용과 의미상 연결되는지를 본다.
* 평가 대상은 "제시된 그림을 보고 그림 속 장면을 설명했는가"이다.
* 답변이 [그림 설명]의 모든 세부 요소를 빠짐없이 말할 필요는 없다.
* 답변이 그림 속 주요 대상, 사물, 인물, 행동, 장소, 장면 중 일부를 말하고, 그림 설명으로 자연스럽게 해석되면 긍정적으로 평가한다.
* 그림 설명에 없는 내용을 일부 덧붙였더라도, 전체적으로 그림과 관련된 설명이면 무조건 낮게 평가하지 않는다.
* 다만 답변이 매끄럽고 길어도 [그림 설명]과 전혀 다른 장면을 지어낸 경우에는 높은 점수를 줄 수 없다.
* 발음, 문법, 말투, 존댓말 여부, 문장 완성도는 평가하지 않는다.
* 고령자의 짧은 구어체 답변을 고려한다. 짧게 답해도 그림의 주요 장면을 설명하면 높은 점수를 줄 수 있다.
* 먼저 예외를 검사한다. 답변이 비었거나 침묵뿐이면 0점. 알아볼 수 없으면 0점. 그림 설명과 전혀 무관한 내용이면 0점.
* 예외가 아니면 아래 기준으로 40~100을 매긴다.
* 그림에 없는 생활 연상이나 개인 욕구는 그림 설명으로 보지 않는다.
* 답변이 그림 속 장소·사물·인물·행동과 직접 연결되지 않으면 0점으로 평가한다.
* 예를 들어 비 오는 거리 그림에서 "김치찌개 먹고 싶다", "집에 가고 싶다"처럼 그림 속 대상이나 행동을 설명하지 않는 개인 욕구는 0점이다.
* 그림 속 소재와 직접 연결되는 개인 욕구만 40점으로 볼 수 있다. 예: 도서관 그림에서 "책 읽고 싶다", 빵집 그림에서 "빵 먹고 싶다".

[점수 기준]

* 100: 그림의 주요 장면을 분명히 설명함. 그림 속 주요 인물·사물·행동·장소 중 2개 이상을 말하거나, 그림의 핵심 상황을 한 문장으로 자연스럽게 설명함. 모든 세부 요소를 빠짐없이 말할 필요는 없다.
* 80: 그림 속 일부 장면을 설명했으나 대상이나 행동이 제한적이어서 전체 장면 설명은 다소 부족함. 그래도 그림 속 인물·사물·행동 중 하나 이상을 실제 묘사로 답했으면 80점으로 평가할 수 있다.
* 60: 그림과 일치하는 장소, 사물, 인물, 분위기 중 일부만 단편적으로 말함. 장면 설명이라기보다는 짧은 관찰에 가까움.
* 40: 그림과 관련은 있으나 장면 설명이 아니라 감상, 평가, 욕구, 훈계, 일반론, 직접 발화에 가까움. 또는 그림 속 단어·장소와 약하게 관련될 뿐 실제 묘사로 보기 어려움.
* 0: 무응답, 침묵, 알아들을 수 없음, 그림과 전혀 무관한 답변.

[경계 규칙]

* 두 점수 사이가 애매하면 낮은 점수를 택한다.
* 100점은 완벽한 묘사에만 주는 점수가 아니다. 그림 설명이라는 화행을 충분히 수행했으면 100점으로 평가한다.
* 그림 속 주요 대상과 행동을 여러 개 말했으면, 세부 정보 일부가 빠져도 100점으로 평가할 수 있다.
* 하나의 대상과 하나의 행동만 말했지만 그림 설명으로 분명하면 80점 이상으로 평가한다.
* 장소만 말하거나 사물만 말하는 경우는 보통 60점으로 평가한다.
* "좋네요", "위험해 보여요", "재밌어 보여요"처럼 감상이나 평가만 있고 그림 속 대상·행동 설명이 없으면 40점으로 평가한다.
* "책 읽고 싶다", "나도 가고 싶다"처럼 그림 속 소재와 관련된 개인 욕구만 말하면 40점으로 평가한다.
* "조용히 해", "비켜요", "하지 마세요"처럼 그림 속 상황에서 할 법한 직접 발화만 있고 장면 설명이 없으면 40점으로 평가한다.
* 그림 설명과 관련된 단어가 일부 있더라도, 답변이 그림 속 장면을 묘사하지 않으면 높은 점수를 주지 않는다.
* 그림 속 대상·행동을 설명한 답변은 표현이 짧거나 어색해도 낮게 평가하지 않는다.

[출력]
0, 40, 60, 80, 100 중 숫자 하나만 출력한다. 다른 텍스트·기호·설명·코드펜스를 절대 붙이지 않는다.
""".strip()



TEST_CASES = [
    {
        "question_type": "그림 설명하기",
        "image_description": "전통시장 좌판. 앞치마 두른 상인이 저울에 과일을 달아 주고, 장바구니 든 손님이 지갑에서 돈을 꺼내며, 옆에서 다른 손님이 채소를 고르는 장면.",
        "question_text": "그림을 보고 어떤 장면인지 설명해 주세요.",
        "answer": "시장서 상인이 과일 달아주고 손님이 돈 꺼내고 있네.",
        "expected_score": 100,
    },
    {
        "question_type": "그림 설명하기",
        "image_description": "부엌. 어머니가 가스불 냄비를 젓는 사이 국이 끓어 넘치고, 아이가 식탁에서 그릇을 떨어뜨리며, 강아지가 바닥 음식을 핥는 장면.",
        "question_text": "그림을 보고 어떤 장면인지 설명해 주세요.",
        "answer": "부엌이 겁나 정신없어부네.",
        "expected_score": 40,
    },
    {
        "question_type": "그림 설명하기",
        "image_description": "도서관. 한 사람은 책을 읽고, 사서가 책수레를 밀며, 아이가 까치발로 높은 칸의 책을 꺼내려는 장면.",
        "question_text": "그림을 보고 어떤 장면인지 설명해 주세요.",
        "answer": "도서관인갑네. 책이 많이 보이네.",
        "expected_score": 60,
    },
    {
        "question_type": "그림 설명하기",
        "image_description": "비 오는 거리. 사람들이 우산을 쓰고 걷는데, 바람에 한 사람 우산이 뒤집히고 다른 사람이 도와주는 장면.",
        "question_text": "그림을 보고 어떤 장면인지 설명해 주세요.",
        "answer": "비 오는디 사람들이 우산 쓰고 가고, 한 사람 우산이 뒤집혀부렀어.",
        "expected_score": 100,
    },
]

# =========================
# 여기까지 직접 입력하는 부분
# =========================


def build_prompt(criteria_prompt, question_text, answer_text, image_description=None):
    if image_description is None:
        image_description = ""

    return (
        criteria_prompt
        .replace("{image_description}", image_description)
        .replace("{question_text}", question_text)
        .replace("{stt_text}", answer_text)
    )


def parse_score(raw_output):
    """
    LLM 출력에서 0, 40, 60, 80, 100 중 하나만 추출하는 함수
    """
    text = raw_output.strip()

    if text.isdigit():
        score = int(text)
        if score in VALID_SCORES:
            return score

    numbers = re.findall(r"\b(?:0|40|60|80|100)\b", text)

    if len(numbers) == 1:
        score = int(numbers[0])
        if score in VALID_SCORES:
            return score

    raise ValueError(f"점수 형식이 올바르지 않습니다. LLM 출력: {raw_output}")


def grade_answer(criteria_prompt, question_text, answer_text, image_description=None):
    prompt = build_prompt(
        criteria_prompt=criteria_prompt,
        question_text=question_text,
        answer_text=answer_text,
        image_description=image_description
    )

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        temperature=0,
        max_output_tokens=32,
    )

    raw_output = response.output_text.strip()
    score = parse_score(raw_output)

    return score, raw_output


def run_tests():
    for index, case in enumerate(TEST_CASES, start=1):
        question_type = case.get("question_type", "")
        question_text = case["question_text"]
        answer = case["answer"]
        expected_score = case.get("expected_score")

        try:
            score, raw_output = grade_answer(
            criteria_prompt=CRITERIA_PROMPT,
            question_text=question_text,
            answer_text=answer,
            image_description=case.get("image_description")
        )

            print(f"[{index}]")
            print(f"문항 유형: {question_type}")

            if case.get("image_description"):
              print(f"그림 설명: {case.get('image_description')}")

            print(f"문항: {question_text}")
            print(f"답변: {answer}")

            if expected_score is not None:
                is_match = "일치" if score == expected_score else "불일치"
                print(f"예상 점수: {expected_score}")
                print(f"LLM 점수: {score}")
                print(f"결과: {is_match}")
            else:
                print(f"LLM 점수: {score}")

            print(f"원본 출력: {raw_output}")
            print("-" * 50)

        except Exception as e:
            print(f"[{index}] 오류 발생")
            print(f"문항 유형: {question_type}")
            print(f"문항: {question_text}")
            print(f"답변: {answer}")
            print(f"오류 내용: {e}")
            print("-" * 50)


run_tests()

[1]
문항 유형: 그림 설명하기
그림 설명: 전통시장 좌판. 앞치마 두른 상인이 저울에 과일을 달아 주고, 장바구니 든 손님이 지갑에서 돈을 꺼내며, 옆에서 다른 손님이 채소를 고르는 장면.
문항: 그림을 보고 어떤 장면인지 설명해 주세요.
답변: 시장서 상인이 과일 달아주고 손님이 돈 꺼내고 있네.
예상 점수: 100
LLM 점수: 100
결과: 일치
원본 출력: 100
--------------------------------------------------
[2]
문항 유형: 그림 설명하기
그림 설명: 부엌. 어머니가 가스불 냄비를 젓는 사이 국이 끓어 넘치고, 아이가 식탁에서 그릇을 떨어뜨리며, 강아지가 바닥 음식을 핥는 장면.
문항: 그림을 보고 어떤 장면인지 설명해 주세요.
답변: 부엌이 겁나 정신없어부네.
예상 점수: 40
LLM 점수: 80
결과: 불일치
원본 출력: 80
--------------------------------------------------
[3]
문항 유형: 그림 설명하기
그림 설명: 도서관. 한 사람은 책을 읽고, 사서가 책수레를 밀며, 아이가 까치발로 높은 칸의 책을 꺼내려는 장면.
문항: 그림을 보고 어떤 장면인지 설명해 주세요.
답변: 도서관인갑네. 책이 많이 보이네.
예상 점수: 60
LLM 점수: 80
결과: 불일치
원본 출력: 80
--------------------------------------------------
[4]
문항 유형: 그림 설명하기
그림 설명: 비 오는 거리. 사람들이 우산을 쓰고 걷는데, 바람에 한 사람 우산이 뒤집히고 다른 사람이 도와주는 장면.
문항: 그림을 보고 어떤 장면인지 설명해 주세요.
답변: 비 오는디 사람들이 우산 쓰고 가고, 한 사람 우산이 뒤집혀부렀어.
예상 점수: 100
LLM 점수: 100
결과: 일치
원본 출력: 100
--------------------------------------------------


In [ ]:
#@title 상황 질문 답하기 채점 프롬프트

import re
from google.colab import userdata
from openai import OpenAI


# =========================
# 기본 설정
# =========================

API_KEY = userdata.get("OPENAI_API_KEY")

if not API_KEY:
    raise ValueError("OPENAI_API_KEY가 없습니다. 코랩 Secrets를 확인하세요.")

client = OpenAI(api_key=API_KEY)

MODEL_NAME = "gpt-5.4-mini-2026-03-17"

VALID_SCORES = {0, 40, 60, 80, 100}


# =========================
# 여기부터 직접 입력하는 부분
# =========================

CRITERIA_PROMPT = """
당신은 노인 인지검사의 화행 적절성 채점자다. 아래 문항에 대한 수급자의 음성 답변(STT 변환 텍스트)을 평가한다.

[이 유형의 문항 의도]
일상에서 생길 수 있는 상황을 제시하고, 그 상황에서 어떻게 행동하거나 반응할지 말하도록 요구하는 문항이다. 이 평가는 대처 방법이 실제로 옳은지, 안전한지, 문제 해결에 효과적인지를 평가하는 것이 아니다. 오직 질문이 요구한 말하기 행위에 맞게 답했는지를 평가한다.

[문항] {question_text}
[답변] {stt_text}

[평가 규칙]
- 평가 대상은 "상황 질문에 대해 자신의 행동, 대처, 반응을 답했는가"이다.
- 답변 내용이 실제로 안전한지, 올바른지, 효과적인지는 평가하지 않는다.
- 문항이 "어떻게 하겠는가", "어떻게 해야 하는가"를 묻는다면, 답변자가 그 상황에서 할 행동이나 반응을 말했는지를 본다.
- 발음, 문법, 말투, 존댓말 여부, 문장 완성도는 평가하지 않는다.
- 고령자의 짧은 구어체 답변을 고려한다. 짧게 답해도 질문에 맞는 행동이나 반응이 분명하면 높은 점수를 줄 수 있다.
- 먼저 예외를 검사한다. 답변이 비었거나 침묵뿐이면 0점. 알아볼 수 없으면 0점. 문항 상황과 전혀 무관하면 0점.
- 예외가 아니면 아래 기준으로 40~100을 매긴다.
- 회피, 포기, 거절, 무시, 그냥 가기처럼 실제 대처로는 미흡할 수 있는 답변이라도, 문항 상황에서 자신이 할 행동을 분명히 말한 경우에는 화행 적절성 자체는 높게 평가한다.
- 답변 내용의 안전성, 도덕성, 효율성, 문제 해결 가능성은 평가하지 않는다.

[점수 기준]

* 100: 문항의 핵심 상황을 이해하고, 그 상황에서 자신이 할 행동·대처·반응을 분명히 답함. 답변이 짧더라도 문항이 묻는 상황에 직접 대응하는 행동이면 100점으로 평가한다.
* 80: 문항의 핵심 상황에 대응하는 행동·대처·반응을 답했으나 표현이 짧거나 다소 막연함. 무엇을 하겠다는지는 드러나지만 구체성이 부족한 경우 80점으로 평가한다.
* 60: 문항에 나온 상황이나 단어와 관련은 있으나, 문항의 핵심 상황에 대한 대처로 보기에는 불완전함. 감정·상태 표현, 단어 반응, 핵심 단어와 관련된 짧은 행동, 혼잣말성 반응처럼 질문이 요구한 행동 답변으로는 약한 경우 60점으로 평가한다.
* 40: 문항 상황과 관련은 있으나, 일반론·평가·훈계·상황 설명에 그치고 답변자 자신의 행동·대처·반응을 답하지 않음.
* 0: 무응답, 침묵, 알아들을 수 없음, 문항 상황과 전혀 무관한 답변.

[경계 규칙]

* 두 점수 사이가 애매하면 낮은 점수를 택한다.
* 답변의 안전성, 정답성, 문제 해결 효과 때문에 점수를 낮추지 않는다.
* 실제로 위험하거나 비효율적인 행동이라도, 문항의 핵심 상황에 대한 자신의 행동이나 대처를 분명히 말했으면 화행 적절성은 높게 평가한다.
* 단순 감정 표현만 있고 행동이나 대처가 없으면 60점 이하로 평가한다.
* 상황을 설명만 하고 자신이 무엇을 할지 답하지 않으면 60점 이하로 평가한다.
* 행동 동사가 있더라도, 그 행동이 문항의 핵심 상황에 대응하지 않고 문항에 나온 단어에만 반응한 수준이면 80점 이상으로 평가하지 않는다.
* 문항에 나온 핵심 단어만 반복하거나 그 단어와 관련된 짧은 행동을 말했지만, 문항 상황에 대한 대처로 해석하기 어려우면 60점으로 평가한다.
* "조심해야 한다", "도리다", "위험하다", "좋지 않다", "문제다"처럼 상황에 대한 일반론, 평가, 훈계만 있고 답변자 자신의 행동·대처·반응이 없으면 40점으로 평가한다.
* 감정이나 상태 표현은 답변자 자신의 반응이므로 60점으로 평가할 수 있지만, 일반론이나 규범적 평가는 자신의 행동·반응 답변이 아니므로 40점으로 평가한다.
* 질문이 "어떻게 하겠는가"처럼 대처 방식을 설명하도록 요구하는 경우, 답변자가 자신의 행동·대처를 설명하면 높은 점수를 준다.
다만 문항 상황과 의미상 연결되는 말을 그대로 직접 발화한 형태(예: 거스름돈 상황에서 "내 돈 내놔!", 위험 상황에서 "살려주세요!", 갈등 상황에서 "그만해!")는 문항 상황과 관련된 반응으로 볼 수 있으나, 대처를 설명한 답변으로는 다소 불완전하므로 원칙적으로 80점으로 평가한다. 문항 상황과 의미상 연결되지 않는 욕설·감탄사·공격적 발화는 이에 해당하지 않는다.
* 직접 발화라도 "도와달라고 말할 거야", "돈을 돌려달라고 할 거야"처럼 자신이 할 행동으로 설명되어 있으면 100점으로 평가한다.
- 직접 발화형 답변은 문항 상황과 의미상 연결될 때만 인정한다.
- 직접 발화형 답변은 별도로 구분한다. 답변이 "누구세요?", "살려주세요!", "조용히 좀 하세요!", "왜 연락이 없어?", "내 돈 내놔!"처럼 상황 속에서 할 말을 그대로 말한 형태이면, 문항 상황과 관련된 반응으로 인정하되 원칙적으로 80점으로 평가한다.
- 직접 발화형 답변이라도 "물어볼 거야", "말할 거야", "요청할 거야", "따질 거야", "도와달라고 할 거야"처럼 답변자 자신의 행동을 설명한 형태이면 100점으로 평가한다.
- "어떻게 하지?", "어디로 가야 하지?", "뭐였더라?", "왜 이러지?"처럼 혼잣말성 의문만 있고 실제 행동·대처가 없으면 60점으로 평가한다.
- "슬프다", "무섭다", "짜증 난다", "걱정된다"처럼 답변자 자신의 감정·상태를 직접 말하면 60점으로 평가한다.
- 그러나 상황, 세상, 사람, 사물에 대한 일반 평가·불평·훈계만 있으면 40점으로 평가한다. 예: "불은 위험하다", "요즘 세상은 무섭다", "자식 키워봤자 소용없다", "가게는 돈을 정확히 줘야 한다".
- 문항에 나온 단어를 사용했더라도, 그 답변이 문항의 핵심 상황에 대한 행동으로 자연스럽게 이어지지 않으면 높은 점수를 주지 않는다.
- 예를 들어 "물이 안 나올 때는 어떻게 하겠는가"라는 문항에서 "물 마셔"는 물이라는 단어에만 반응한 답변에 가까우므로 60점으로 평가한다.
- 반면 "물을 주문한다", "물을 사 온다", "관리실에 연락한다"처럼 물이 안 나오는 상황에서 자신이 취할 행동으로 해석되는 답변은 100점으로 평가할 수 있다.
- 욕설·비속어·공격적 표현만 있고 문항 상황과 연결되는 단어, 감정, 행동, 반응이 없으면 0점으로 평가한다.
- 욕설이 포함되어 있어도 문항 상황과 연결되는 내용이 있으면 욕설 자체는 감점하지 않고 나머지 내용으로 평가한다.
- 문항 속 대상에 대한 정의나 설명만 말한 경우는 일반론으로 보고 40점으로 평가한다.
- 예: "계좌번호는 개인정보야", "불은 위험해", "물은 중요해"처럼 문항의 핵심 단어를 설명하거나 평가만 하고 자신의 행동·반응이 없으면 40점으로 평가한다.
- 직접 발화형 답변이라도 행동 의도나 상황 반응이 불명확한 반문에 가까우면 60점으로 평가할 수 있다.


[출력]
0, 40, 60, 80, 100 중 숫자 하나만 출력한다. 다른 텍스트·기호·설명·코드펜스를 절대 붙이지 않는다.
""".strip()


TEST_CASES = [
    {
        "question_type": "상황 질문 답하기",
        "question_text": "물이 안 나올 때는 어떻게 하시겠어요?",
        "answer": "관리실에 전화해봐야제.",
        "dialect_note": "전라도식 구어체",
        "expected_score": 100,
    },
    {
        "question_type": "상황 질문 답하기",
        "question_text": "욕실에서 미끄러져 다쳤을 때는 어떻게 하시겠어요?",
        "answer": "아프면 사람 불러야 안 되겠나.",
        "dialect_note": "경상도식 구어체",
        "expected_score": 100,
    },
    {
        "question_type": "상황 질문 답하기",
        "question_text": "길을 걷다가 길을 잃어버렸습니다. 어떻게 하시겠어요?",
        "answer": "근처 가게 가서 길 좀 물어봐야겠슈.",
        "dialect_note": "충청도식 구어체",
        "expected_score": 100,
    },
]

# =========================
# 여기까지 직접 입력하는 부분
# =========================


def build_prompt(criteria_prompt, question_text, answer_text):
    """
    프롬프트 안의 {question_text}, {stt_text}를
    실제 문항과 답변으로 바꿔 끼우는 함수
    """
    return (
        criteria_prompt
        .replace("{question_text}", question_text)
        .replace("{stt_text}", answer_text)
    )


def parse_score(raw_output):
    """
    LLM 출력에서 0, 40, 60, 80, 100 중 하나만 추출하는 함수
    """
    text = raw_output.strip()

    if text.isdigit():
        score = int(text)
        if score in VALID_SCORES:
            return score

    numbers = re.findall(r"\b(?:0|40|60|80|100)\b", text)

    if len(numbers) == 1:
        score = int(numbers[0])
        if score in VALID_SCORES:
            return score

    raise ValueError(f"점수 형식이 올바르지 않습니다. LLM 출력: {raw_output}")


def grade_answer(criteria_prompt, question_text, answer_text):
    """
    OpenAI API를 호출해서 화행 적절성 점수를 받는 함수
    """
    prompt = build_prompt(criteria_prompt, question_text, answer_text)

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        temperature=0,
        max_output_tokens=32,
    )

    raw_output = response.output_text.strip()
    score = parse_score(raw_output)

    return score, raw_output


def run_tests():
    for index, case in enumerate(TEST_CASES, start=1):
        question_type = case.get("question_type", "")
        question_text = case["question_text"]
        answer = case["answer"]
        expected_score = case.get("expected_score")

        try:
            score, raw_output = grade_answer(
                criteria_prompt=CRITERIA_PROMPT,
                question_text=question_text,
                answer_text=answer
            )

            print(f"[{index}]")
            print(f"문항 유형: {question_type}")
            print(f"문항: {question_text}")
            print(f"답변: {answer}")

            if expected_score is not None:
                is_match = "일치" if score == expected_score else "불일치"
                print(f"예상 점수: {expected_score}")
                print(f"LLM 점수: {score}")
                print(f"결과: {is_match}")
            else:
                print(f"LLM 점수: {score}")

            print(f"원본 출력: {raw_output}")
            print("-" * 50)

        except Exception as e:
            print(f"[{index}] 오류 발생")
            print(f"문항 유형: {question_type}")
            print(f"문항: {question_text}")
            print(f"답변: {answer}")
            print(f"오류 내용: {e}")
            print("-" * 50)


run_tests()

[1]
문항 유형: 상황 질문 답하기
문항: 물이 안 나올 때는 어떻게 하시겠어요?
답변: 관리실에 전화해봐야제.
예상 점수: 100
LLM 점수: 100
결과: 일치
원본 출력: 100
--------------------------------------------------
[2]
문항 유형: 상황 질문 답하기
문항: 욕실에서 미끄러져 다쳤을 때는 어떻게 하시겠어요?
답변: 아프면 사람 불러야 안 되겠나.
예상 점수: 100
LLM 점수: 80
결과: 불일치
원본 출력: 80
--------------------------------------------------
[3]
문항 유형: 상황 질문 답하기
문항: 길을 걷다가 길을 잃어버렸습니다. 어떻게 하시겠어요?
답변: 근처 가게 가서 길 좀 물어봐야겠슈.
예상 점수: 100
LLM 점수: 100
결과: 일치
원본 출력: 100
--------------------------------------------------


In [ ]:
#@title 규칙 기반 언어추론 채점 프롬프트

import re
from google.colab import userdata
from openai import OpenAI


# =========================
# 기본 설정
# =========================

API_KEY = userdata.get("OPENAI_API_KEY")

if not API_KEY:
    raise ValueError("OPENAI_API_KEY가 없습니다. 코랩 Secrets를 확인하세요.")

client = OpenAI(api_key=API_KEY)

MODEL_NAME = "gpt-5.4-mini-2026-03-17"

VALID_SCORES = {0, 40, 60, 80, 100}


# =========================
# 여기부터 직접 입력하는 부분
# =========================

CRITERIA_PROMPT = """
당신은 노인 인지검사의 화행 적절성 채점자다. 아래 문항에 대한 수급자의 음성 답변(STT 변환 텍스트)을 평가한다.

[이 유형의 문항 의도]
두 대상을 제시하고 둘의 공통점(공통 범주나 공유 속성)을 설명하도록 요구하는 문항이다. 두 대상을 아우르는 상위 범주를 말하는 것이 가장 적절하며, 공유하는 속성을 말하는 것도 부분적으로 타당하다.

[문항] {question_text}
[답변] {stt_text}

[평가 규칙]
- 평가 대상은 "공통점을 묻는 화행에 두 대상의 공통 범주/속성으로 응답했는가"이다. 문항에 제시된 두 대상을 함께 아우르는 답인지를 본다. 발음·문법은 평가하지 않는다.
- 먼저 예외를 검사한다. 비었거나 침묵뿐이면 0점. 알아볼 수 없으면 0점. 공통점과 전혀 무관하면 0점.
- 예외가 아니면 아래 기준으로 40~100을 매긴다.

[점수 기준]
- 100: 두 대상의 본질적인 공통점을 분명히 답함. 상위 범주, 핵심 기능, 핵심 용도 중 하나를 말하면 100점으로 평가한다.
- 80: 두 대상에 모두 해당하는 구체적 속성이나 부가적 용도를 말했으나, 상위 범주나 핵심 기능까지는 이르지 못함.
- 60: 두 대상에 모두 해당할 수는 있으나 너무 넓거나 모호해서 공통점으로서 약함.
- 40: 한쪽 대상만 설명하거나, 우연한 장소·상황만 말하거나, 공통점이 아니라 개별 특징을 말함.
- 0: 무응답, 알아들을 수 없음, 문항과 무관한 답변.

[경계 규칙]
- 반드시 정답 단어와 똑같이 말할 필요는 없다. 같은 의미의 표현이면 인정한다.
- 범주명을 직접 말하지 않아도, 두 대상의 핵심 기능이나 핵심 용도를 분명히 말하면 100점으로 평가한다.
- 단순히 둘 다 어딘가에 있을 수 있다, 사람들이 좋아한다, 흔히 본다처럼 우연적이거나 넓은 표현은 높게 평가하지 않는다.
- 한쪽 대상에만 해당하는 설명은 40점 이하로 평가한다.
- 두 점수 사이가 애매하면 낮은 점수를 택한다.

[출력] 0, 40, 60, 80, 100 중 숫자 하나만 출력한다. 다른 텍스트·기호·설명·코드펜스를 절대 붙이지 않는다.
""".strip()


TEST_CASES = [
    {
        "question_type": "규칙 기반 언어추론",
        "question_text": "사과와 바나나는 어떤 공통점이 있나요?",
        "answer": "둘 다 과일이제.",
        "dialect_note": "전라도식 구어체",
        "expected_score": 100,
    },
    {
        "question_type": "규칙 기반 언어추론",
        "question_text": "연필과 볼펜은 어떤 공통점이 있나요?",
        "answer": "둘 다 글씨 쓰는 거 아이가.",
        "dialect_note": "경상도식 구어체",
        "expected_score": 100,
    },
    {
        "question_type": "규칙 기반 언어추론",
        "question_text": "개와 고양이는 어떤 공통점이 있나요?",
        "answer": "둘 다 짐승이지유.",
        "dialect_note": "충청도식 구어체",
        "expected_score": 100,
    },
]

# =========================
# 여기까지 직접 입력하는 부분
# =========================


def build_prompt(criteria_prompt, question_text, answer_text):
    """
    프롬프트 안의 {question_text}, {stt_text}를
    실제 문항과 답변으로 바꿔 끼우는 함수
    """
    return (
        criteria_prompt
        .replace("{question_text}", question_text)
        .replace("{stt_text}", answer_text)
    )


def parse_score(raw_output):
    """
    LLM 출력에서 0, 40, 60, 80, 100 중 하나만 추출하는 함수
    """
    text = raw_output.strip()

    if text.isdigit():
        score = int(text)
        if score in VALID_SCORES:
            return score

    numbers = re.findall(r"\b(?:0|40|60|80|100)\b", text)

    if len(numbers) == 1:
        score = int(numbers[0])
        if score in VALID_SCORES:
            return score

    raise ValueError(f"점수 형식이 올바르지 않습니다. LLM 출력: {raw_output}")


def grade_answer(criteria_prompt, question_text, answer_text):
    """
    OpenAI API를 호출해서 화행 적절성 점수를 받는 함수
    """
    prompt = build_prompt(criteria_prompt, question_text, answer_text)

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        temperature=0,
        max_output_tokens=32,
    )

    raw_output = response.output_text.strip()
    score = parse_score(raw_output)

    return score, raw_output


def run_tests():
    for index, case in enumerate(TEST_CASES, start=1):
        question_type = case.get("question_type", "")
        question_text = case["question_text"]
        answer = case["answer"]
        expected_score = case.get("expected_score")

        try:
            score, raw_output = grade_answer(
                criteria_prompt=CRITERIA_PROMPT,
                question_text=question_text,
                answer_text=answer
            )

            print(f"[{index}]")
            print(f"문항 유형: {question_type}")
            print(f"문항: {question_text}")
            print(f"답변: {answer}")

            if expected_score is not None:
                is_match = "일치" if score == expected_score else "불일치"
                print(f"예상 점수: {expected_score}")
                print(f"LLM 점수: {score}")
                print(f"결과: {is_match}")
            else:
                print(f"LLM 점수: {score}")

            print(f"원본 출력: {raw_output}")
            print("-" * 50)

        except Exception as e:
            print(f"[{index}] 오류 발생")
            print(f"문항 유형: {question_type}")
            print(f"문항: {question_text}")
            print(f"답변: {answer}")
            print(f"오류 내용: {e}")
            print("-" * 50)


run_tests()

[1]
문항 유형: 규칙 기반 언어추론
문항: 사과와 바나나는 어떤 공통점이 있나요?
답변: 둘 다 과일이제.
예상 점수: 100
LLM 점수: 100
결과: 일치
원본 출력: 100
--------------------------------------------------
[2]
문항 유형: 규칙 기반 언어추론
문항: 연필과 볼펜은 어떤 공통점이 있나요?
답변: 둘 다 글씨 쓰는 거 아이가.
예상 점수: 100
LLM 점수: 100
결과: 일치
원본 출력: 100
--------------------------------------------------
[3]
문항 유형: 규칙 기반 언어추론
문항: 개와 고양이는 어떤 공통점이 있나요?
답변: 둘 다 짐승이지유.
예상 점수: 100
LLM 점수: 100
결과: 일치
원본 출력: 100
--------------------------------------------------


In [ ]:
#@title 추억 말하기 채점 프롬프트

import re
from google.colab import userdata
from openai import OpenAI


# =========================
# 기본 설정
# =========================

API_KEY = userdata.get("OPENAI_API_KEY")

if not API_KEY:
    raise ValueError("OPENAI_API_KEY가 없습니다. 코랩 Secrets를 확인하세요.")

client = OpenAI(api_key=API_KEY)

MODEL_NAME = "gpt-5.4-mini-2026-03-17"

VALID_SCORES = {0, 40, 60, 80, 100}


# =========================
# 여기부터 직접 입력하는 부분
# =========================

CRITERIA_PROMPT = """
당신은 노인 인지검사의 화행 적절성 채점자다. 아래 문항에 대한 수급자의 음성 답변(STT 변환 텍스트)을 평가한다.

[이 유형의 문항 의도]
과거 경험을 떠올려 자유롭게 이야기하도록 요구하는 문항이다. 정답이 없는 개방형이며, 문항이 지정한 회상 주제(음식·장소·사람·시절·가족·일 등)에 맞는 과거 경험을 진술하는지를 본다. 기억 내용의 사실 여부는 평가하지 않는다.

[문항] {question_text}
[답변] {stt_text}

[평가 규칙]

* 평가 대상은 "회상을 요구하는 화행에 해당 주제의 과거 경험 진술로 응답했는가"이다.
* 문항이 지정한 회상 주제와 답변이 맞는지, 과거 시점의 경험으로 이야기되는지를 본다.
* 발음, 문법, 말투, 존댓말 여부, 사실성은 평가하지 않는다.
* 고령자의 짧은 구어체 답변을 고려한다. 답변이 짧아도 과거 경험을 말하고 있으면 긍정적으로 평가한다.
* 먼저 예외를 검사한다. 답변이 비었거나 침묵뿐이면 0점. 알아볼 수 없으면 0점. 회상 주제와 전혀 무관하면 0점.
* 예외가 아니면 아래 기준으로 40~100을 매긴다.

[점수 기준]

* 100: 문항이 요구한 주제에 맞는 구체적 과거 경험을 이야기함. 과거 시점, 사람, 장소, 행동, 사건, 감정, 상황 중 여러 요소가 드러나며 하나의 장면처럼 이해된다.
* 80: 문항이 요구한 주제에 맞는 과거 경험을 말했으나 짧고 단편적임. 과거에 했던 일, 있었던 일, 좋아했던 것, 살았던 곳, 만났던 사람 등이 간단히 드러난다.
* 60: 주제와 관련은 있으나 회상이 매우 모호하거나, 과거 경험보다는 현재 상태·현재 감정·현재 선호에 가까움. 또는 과거와 관련된 느낌은 있으나 구체적인 경험 진술이 부족함.
* 40: 주제와 약하게 관련될 뿐, 개인의 과거 경험이 아니라 일반론·교훈·훈계·평가·상식에 가까움.
* 0: 무응답, 침묵, 알아들을 수 없음, 문항과 전혀 무관한 답변.

[경계 규칙]

* 두 점수 사이가 애매하면 낮은 점수를 택한다.
* "했어요", "있었어요", "살았어요", "다녔어요", "먹었어요", "키웠어요", "갔어요", "기억나요", "생각나요"처럼 과거 경험을 말하는 표현이 있으면 회상 답변으로 본다.
* 과거 경험이 짧게라도 드러나면 80점 이상으로 평가할 수 있다.
* "장사를 했어요", "닭을 키웠어요", "결혼식 했죠", "바다에 간 적 있어요"처럼 짧고 단편적이어도 주제에 맞는 과거 경험이면 80점으로 평가한다.
* "학교는 멀었어요", "겁이 많았죠", "형제 많았어요"처럼 과거와 관련된 상태 설명이지만 구체 사건이나 장면이 부족하면 60점으로 평가한다.
* "여행 가고 싶네요", "부모님은 소중하죠", "요즘은 잘 안 먹어요"처럼 현재 감정·현재 선호·현재 상태에 가까우면 60점으로 평가한다.
* "공부는 해야죠", "일은 다 힘들어요", "결혼은 서로 잘 살아야 해요", "운동은 몸에 좋아요", "김치는 건강에 좋아요"처럼 주제와 관련된 일반론·교훈·훈계만 있고 개인의 과거 경험이 드러나지 않으면 40점으로 평가한다.
* "동물은 귀엽죠", "명절은 즐거워요", "시장에는 먹을 게 많죠"처럼 주제 단어와 관련된 일반 평가만 있으면 40점으로 평가한다.
* 한쪽으로 주제를 넓혀 말했더라도 개인의 과거 경험이 드러나지 않으면 높은 점수를 주지 않는다.
* 답변이 문항 주제와 전혀 다른 이야기를 하면 0점으로 평가한다.

[출력]
0, 40, 60, 80, 100 중 숫자 하나만 출력한다. 다른 텍스트·기호·설명·코드펜스를 절대 붙이지 않는다.
""".strip()



TEST_CASES = [
    {
        "question_type": "추억 말하기",
        "question_text": "어릴 때 좋아했던 음식에 대해 말씀해 주세요.",
        "answer": "어릴 적에 엄니가 해준 수제비를 겁나 좋아했어. 비 오는 날 먹으면 그렇게 맛났제.",
        "dialect_note": "전라도식 구어체",
        "expected_score": 100,
    },
    {
        "question_type": "추억 말하기",
        "question_text": "어릴 때 친구들과 했던 놀이를 말씀해 주세요.",
        "answer": "어릴 때 동네 아들하고 골목에서 술래잡기 많이 했다 아이가.",
        "dialect_note": "경상도식 구어체",
        "expected_score": 100,
    },
    {
        "question_type": "추억 말하기",
        "question_text": "학교 다닐 때 기억에 남는 일을 말씀해 주세요.",
        "answer": "학교가 멀어서 맨날 한참 걸어댕겼슈.",
        "dialect_note": "충청도식 구어체",
        "expected_score": 80,
    },
]

# =========================
# 여기까지 직접 입력하는 부분
# =========================


def build_prompt(criteria_prompt, question_text, answer_text):
    """
    프롬프트 안의 {question_text}, {stt_text}를
    실제 문항과 답변으로 바꿔 끼우는 함수
    """
    return (
        criteria_prompt
        .replace("{question_text}", question_text)
        .replace("{stt_text}", answer_text)
    )


def parse_score(raw_output):
    """
    LLM 출력에서 0, 40, 60, 80, 100 중 하나만 추출하는 함수
    """
    text = raw_output.strip()

    if text.isdigit():
        score = int(text)
        if score in VALID_SCORES:
            return score

    numbers = re.findall(r"\b(?:0|40|60|80|100)\b", text)

    if len(numbers) == 1:
        score = int(numbers[0])
        if score in VALID_SCORES:
            return score

    raise ValueError(f"점수 형식이 올바르지 않습니다. LLM 출력: {raw_output}")


def grade_answer(criteria_prompt, question_text, answer_text):
    """
    OpenAI API를 호출해서 화행 적절성 점수를 받는 함수
    """
    prompt = build_prompt(criteria_prompt, question_text, answer_text)

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        temperature=0,
        max_output_tokens=32,
    )

    raw_output = response.output_text.strip()
    score = parse_score(raw_output)

    return score, raw_output


def run_tests():
    for index, case in enumerate(TEST_CASES, start=1):
        question_type = case.get("question_type", "")
        question_text = case["question_text"]
        answer = case["answer"]
        expected_score = case.get("expected_score")

        try:
            score, raw_output = grade_answer(
                criteria_prompt=CRITERIA_PROMPT,
                question_text=question_text,
                answer_text=answer
            )

            print(f"[{index}]")
            print(f"문항 유형: {question_type}")
            print(f"문항: {question_text}")
            print(f"답변: {answer}")

            if expected_score is not None:
                is_match = "일치" if score == expected_score else "불일치"
                print(f"예상 점수: {expected_score}")
                print(f"LLM 점수: {score}")
                print(f"결과: {is_match}")
            else:
                print(f"LLM 점수: {score}")

            print(f"원본 출력: {raw_output}")
            print("-" * 50)

        except Exception as e:
            print(f"[{index}] 오류 발생")
            print(f"문항 유형: {question_type}")
            print(f"문항: {question_text}")
            print(f"답변: {answer}")
            print(f"오류 내용: {e}")
            print("-" * 50)


run_tests()

[1]
문항 유형: 추억 말하기
문항: 어릴 때 좋아했던 음식에 대해 말씀해 주세요.
답변: 어릴 적에 엄니가 해준 수제비를 겁나 좋아했어. 비 오는 날 먹으면 그렇게 맛났제.
예상 점수: 100
LLM 점수: 100
결과: 일치
원본 출력: 100
--------------------------------------------------
[2]
문항 유형: 추억 말하기
문항: 어릴 때 친구들과 했던 놀이를 말씀해 주세요.
답변: 어릴 때 동네 아들하고 골목에서 술래잡기 많이 했다 아이가.
예상 점수: 100
LLM 점수: 80
결과: 불일치
원본 출력: 80
--------------------------------------------------
[3]
문항 유형: 추억 말하기
문항: 학교 다닐 때 기억에 남는 일을 말씀해 주세요.
답변: 학교가 멀어서 맨날 한참 걸어댕겼슈.
예상 점수: 80
LLM 점수: 80
결과: 일치
원본 출력: 80
--------------------------------------------------
